# Esta parte do código se refere à pipeline da camada SILVER em BATCH para testes antes de subir ao AWS

In [33]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [34]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
import time
import os
from pathlib import Path
from datetime import datetime

import pandas as pd

In [35]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
DATA_BRONZE = Path("bronze")
DATA_SILVER = Path("silver")

DATA_SILVER.mkdir(parents=True, exist_ok=True)

INGESTION_DATE = datetime.now().strftime("%Y-%m-%d")

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos",
    "pib_municipio",
    "indicadores_educacionais_municipio",
    "populacao_municipio",
    "inse_escola"
]

In [36]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [37]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA SILVER")
log.info("~" * 35)

2026-08-25 07:39:59,462 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-25 07:39:59,463 | INFO     | INICIANDO ETL DA CAMADA SILVER
2026-08-25 07:39:59,463 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [38]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# (mesmo padrão usado na camada Bronze, para manter consistência de logs
# entre as camadas)

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro (sempre em nível ERROR no log) e tenta
    publicar em um tópico SNS, se configurado via variável de ambiente
    SNS_TOPIC_ARN, para que a falha não dependa de alguém checar o log
    manualmente.
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [39]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA BRONZE
"""
    Lê o arquivo Parquet mais recente da camada Bronze para uma tabela.

    A Bronze agora particiona por ingestion_date (bronze/{tabela}/
    ingestion_date={data}/{tabela}.parquet), preservando o histórico de
    todas as cargas. A Silver processa, por padrão, a partição mais
    recente (a última carga executada).

    Args:
        tabela (str): Nome da tabela.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_bronze(tabela):

    pasta_tabela = DATA_BRONZE / tabela

    particoes = sorted(pasta_tabela.glob("ingestion_date=*"))

    if not particoes:
        raise FileNotFoundError(
            f"Nenhuma partição de ingestion_date encontrada em {pasta_tabela}. "
            f"Rode a camada Bronze antes da Silver."
        )

    particao_mais_recente = particoes[-1]
    caminho = particao_mais_recente / f"{tabela}.parquet"

    log.info(f"Lendo a camada Bronze (partição mais recente): {caminho}")

    return pd.read_parquet(caminho)

In [40]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE CHECKS 
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {

    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","sigla_uf","serie","rede"], "critico": False},
    ],

    "municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","id_municipio","serie","rede"], "critico": False},
    ],

    "meta_alfabetizacao_brasil": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","sigla_uf","rede"], "critico": False},
    ],

    "meta_alfabetizacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","id_municipio","rede"], "critico": False}
    ],

    "alunos": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "id_aluno", "critico": True},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
    ],

    "pib_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "indicadores_educacionais_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "populacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "populacao", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "inse_escola": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_escola"], "critico": False},
    ]
}

In [41]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CHECANDO A QUALIDADE DOS DADOS DA SILVER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):
    """
    Executa as validações de qualidade da camada Silver.

    Args:
        df (pandas.DataFrame): DataFrame da Silver.
        checks (list): Lista de regras de validação.

    Raises:
        Exception: Caso alguma validação crítica falhe.
    """

    log.info(f"[DQ:SILVER] Iniciando verificações ({len(checks)} regra(s))")

    passou = 0
    falhou = 0
    criticos = 0

    for check in checks:

        tipo = check["tipo"]
        coluna = check.get("coluna")
        valor = check.get("valor")
        critico = check.get("critico", True)

        ok = False
        detalhe = ""

        try:

            if tipo == "not_null":

                nulos = df[coluna].isnull().sum()

                ok = nulos == 0
                detalhe = f"{nulos} nulos encontrados"

            elif tipo == "min_count":

                contagem = len(df)

                ok = contagem >= valor
                detalhe = f"contagem={contagem} | mínimo={valor}"

            elif tipo == "unique":

                duplicados = df.duplicated(subset=coluna).sum()

                ok = duplicados == 0
                detalhe = f"{duplicados} duplicados encontrados"

        except Exception as e:

            ok = False
            detalhe = str(e)

        status = "PASS" if ok else ("FAIL" if critico else "WARN")

        if ok:

            passou += 1
            log.info(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

        else:

            falhou += 1

            if critico:

                criticos += 1
                log.error(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

            else:

                log.warning(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

    score = round((passou / len(checks)) * 100, 1)

    log.info(f"[DQ:SILVER] Score={score}% | PASS={passou} | FAIL={falhou}")

    if criticos > 0:

        raise Exception(
            f"[DQ:SILVER] {criticos} validação(ões) crítica(s) falharam."
        )

In [42]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# VALIDAÇÃO ENTRE TABELAS: CHAVES DE RELACIONAMENTO E CONSISTÊNCIA
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Diferente de `checar_qualidade` (que valida uma tabela sozinha), estas
# funções validam o RELACIONAMENTO entre duas tabelas -- só são possíveis
# agora que existe a integração alunos + município + UF.

def checar_integridade_referencial(df, coluna_fk, df_referencia, coluna_referencia, nome_relacao, critico=True):
    """
    Verifica se todo valor não nulo de `coluna_fk` em `df` existe em
    `coluna_referencia` de `df_referencia` (chave estrangeira).
    Ex.: todo id_municipio de `alunos` deve existir na tabela `municipio`.

    Args:
        df (pandas.DataFrame): Tabela "filha" (contém a chave estrangeira).
        coluna_fk (str): Coluna de `df` que deveria referenciar a outra tabela.
        df_referencia (pandas.DataFrame): Tabela "pai" (contém a chave primária/domínio).
        coluna_referencia (str): Coluna de `df_referencia` que serve de domínio válido.
        nome_relacao (str): Nome descritivo da relação, usado no log.
        critico (bool): Se True, levanta exceção quando houver órfãos.

    Raises:
        AssertionError: Se `critico=True` e existirem valores órfãos.
    """
    chaves_validas = set(df_referencia[coluna_referencia].dropna())
    valores = df[coluna_fk].dropna()

    orfaos = valores[~valores.isin(chaves_validas)]
    qtd_orfaos = len(orfaos)
    qtd_total = len(valores)
    percentual = round((qtd_orfaos / qtd_total) * 100, 2) if qtd_total else 0.0

    ok = qtd_orfaos == 0
    status = "PASS" if ok else ("FAIL" if critico else "WARN")
    detalhe = (
        f"{qtd_orfaos} de {qtd_total} ({percentual}%) valor(es) de '{coluna_fk}' "
        f"sem correspondência em '{nome_relacao}.{coluna_referencia}'"
    )

    if ok:
        log.info(f"[DQ:SILVER:REFERENCIAL] {status} | {detalhe}")
    elif critico:
        log.error(f"[DQ:SILVER:REFERENCIAL] {status} | {detalhe}")
        raise AssertionError(detalhe)
    else:
        log.warning(f"[DQ:SILVER:REFERENCIAL] {status} | {detalhe}")

    return {"ok": ok, "orfaos": qtd_orfaos, "total": qtd_total, "percentual": percentual}


def checar_consistencia_territorial(df_alunos_integrado, critico=False):
    """
    Verifica a CONSISTÊNCIA entre a UF derivada do município do aluno
    (`sigla_uf`, calculada via CODIGO_UF_PARA_SIGLA) e o contexto estadual
    trazido pelo merge com a tabela `uf` (`sigla_uf_nome`).

    Se `sigla_uf` está preenchida mas `sigla_uf_nome` ficou nula após o
    merge, é sinal de inconsistência entre as bases: a UF derivada não
    encontrou correspondência na tabela `uf` para aquele ano/rede -- ou
    seja, as duas fontes territoriais "discordam" para aquele registro.

    Args:
        df_alunos_integrado (pandas.DataFrame): Resultado de
            construir_silver_alunos_integrado.
        critico (bool): Se True, levanta exceção quando houver inconsistências.

    Raises:
        AssertionError: Se `critico=True` e existirem inconsistências.
    """
    tem_sigla = df_alunos_integrado["sigla_uf"].notna()
    tem_nome = df_alunos_integrado["sigla_uf_nome"].notna()

    inconsistentes = df_alunos_integrado[tem_sigla & ~tem_nome]
    qtd = len(inconsistentes)
    total = tem_sigla.sum()
    percentual = round((qtd / total) * 100, 2) if total else 0.0

    ok = qtd == 0
    status = "PASS" if ok else ("FAIL" if critico else "WARN")
    detalhe = (
        f"{qtd} de {total} ({percentual}%) aluno(s) com sigla_uf derivada mas "
        f"sem correspondência em uf.sigla_uf para o mesmo ano/rede"
    )

    if ok:
        log.info(f"[DQ:SILVER:CONSISTENCIA] {status} | {detalhe}")
    elif critico:
        log.error(f"[DQ:SILVER:CONSISTENCIA] {status} | {detalhe}")
        raise AssertionError(detalhe)
    else:
        log.warning(f"[DQ:SILVER:CONSISTENCIA] {status} | {detalhe}")

    return {"ok": ok, "inconsistentes": qtd, "total": total, "percentual": percentual}

In [43]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SCHEMA DE TIPOS POR TABELA (conversão de tipo da Silver)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

_COLUNAS_META = {f"meta_alfabetizacao_20{ano}": "float64" for ano in range(24, 31)}
_COLUNAS_PROPORCAO = {f"proporcao_aluno_nivel_{n}": "float64" for n in range(0, 9)}

TIPOS_COLUNAS = {
    "uf": {
        "ano": "Int64", "sigla_uf": "string", "sigla_uf_nome": "string",
        "serie": "string", "rede": "string",
        "taxa_alfabetizacao": "float64", "media_portugues": "float64",
        **_COLUNAS_PROPORCAO,
    },
    "municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "serie": "string", "rede": "string",
        "taxa_alfabetizacao": "float64", "media_portugues": "float64",
        **_COLUNAS_PROPORCAO,
    },
    "meta_alfabetizacao_brasil": {
        "ano": "Int64", "rede": "string", "taxa_alfabetizacao": "float64",
        "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "meta_alfabetizacao_uf": {
        "ano": "Int64", "sigla_uf": "string", "sigla_uf_nome": "string",
        "rede": "string", "taxa_alfabetizacao": "float64",
        "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "meta_alfabetizacao_municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "rede": "string", "taxa_alfabetizacao": "float64",
        "nivel_alfabetizacao": "string", "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "alunos": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "id_escola": "Int64", "id_aluno": "Int64",
        "caderno": "string", "serie": "string", "rede": "string",
        "presenca": "string", "preenchimento_caderno": "string",
        "alfabetizado": "string", "proficiencia": "float64", "peso_aluno": "float64",
    },
    "pib_municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "pib": "float64", "impostos_liquidos": "float64", "va": "float64",
        "va_agropecuaria": "float64", "va_industria": "float64",
        "va_servicos": "float64", "va_adespss": "float64",
    },
    "indicadores_educacionais_municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "localizacao": "string", "rede": "string",
        "atu_ef_anos_iniciais": "float64", "had_ef_anos_iniciais": "float64",
        "tdi_ef_2_ano": "float64", "taxa_aprovacao_ef_2_ano": "float64",
        "taxa_reprovacao_ef_2_ano": "float64", "taxa_abandono_ef_2_ano": "float64",
        "dsu_ef_anos_iniciais": "float64", "afd_ef_anos_iniciais_grupo_1": "float64",
        "ird_alta": "float64", "ird_baixa_regularidade": "float64",
        "icg_nivel_1": "float64", "icg_nivel_2": "float64", "icg_nivel_3": "float64",
        "icg_nivel_4": "float64", "icg_nivel_5": "float64", "icg_nivel_6": "float64",
    },
    "populacao_municipio": {
        "ano": "Int64", "sigla_uf": "string", "sigla_uf_nome": "string",
        "id_municipio": "Int64", "id_municipio_nome": "string", "populacao": "Int64",
    },
    "inse_escola": {
        "ano": "Int64", "id_municipio": "Int64", "id_escola": "Int64",
        "inse": "float64", "classificacao": "string",
    },
}

# Colunas que funcionam como chave de junção entre tabelas e cujo *formato*
# (não o conteúdo/categoria) precisa ser padronizado para o merge funcionar
# de forma confiável. `rede`/`serie` NÃO entram aqui: o notebook da Gold
# faz comparação de string literal (ex.: `df["rede"] != "Total (...)"`),
# então alterar caixa/formatação delas quebraria esse filtro silenciosamente.
COLUNAS_CHAVE_NORMALIZAR = ["sigla_uf"]


def aplicar_conversao_tipo(df, tabela):
    """
    Converte cada coluna para o tipo declarado em TIPOS_COLUNAS, evitando que
    chaves de junção (ano, id_municipio) fiquem com tipos diferentes entre
    tabelas (ex.: int64 numa tabela e float64/string noutra), o que faz o
    merge falhar silenciosamente (não dá erro, só não casa nenhuma linha).
    """
    tipos = TIPOS_COLUNAS.get(tabela, {})

    for coluna, tipo in tipos.items():

        if coluna not in df.columns:
            continue

        try:
            if tipo in ("Int64", "float64"):
                df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype(tipo)
            else:
                df[coluna] = df[coluna].astype("string")
        except Exception as e:
            log.warning(f"[SILVER] {tabela}: falha ao converter '{coluna}' para {tipo}: {e}")

    return df


def normalizar_chaves_join(df, tabela):
    """
    Padroniza o formato (maiúsculas/espaços) das colunas-chave de junção
    territoriais. Mantém o conteúdo de colunas categóricas de negócio
    (rede, serie) intacto, pois a Gold depende do texto exato delas.
    """
    for coluna in COLUNAS_CHAVE_NORMALIZAR:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.strip().str.upper()

    return df

In [44]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# COLUNAS ESSENCIAIS POR TABELA (usadas SÓ no tratamento de nulos)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
COLUNAS_ESSENCIAIS = {
    "uf": ["ano", "sigla_uf"],
    "municipio": ["ano", "id_municipio"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio"],
    "alunos": ["id_aluno", "id_escola", "id_municipio"],
    "pib_municipio": ["ano", "id_municipio"],
    "indicadores_educacionais_municipio": ["ano", "id_municipio"],
    "populacao_municipio": ["ano", "id_municipio"],
    "inse_escola": ["ano", "id_escola", "id_municipio"],
}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CHAVE DE NEGÓCIO POR TABELA (usada SÓ no dedup)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# IMPORTANTE: esta chave precisa ser a combinação COMPLETA de colunas que
# identifica um registro de negócio único -- não pode ser um subconjunto
# "otimista" como COLUNAS_ESSENCIAIS acima. As tabelas territoriais (uf,
# municipio e as duas tabelas de meta) têm uma linha por combinação de
# ano + território + rede (e, em uf/municipio, também serie). Usar só
# ano+sigla_uf (por exemplo) faz o dedup colapsar Municipal/Estadual/
# Federal/Privada da mesma UF/ano em uma linha só -- perda real de dado,
# não remoção de duplicata. Esta chave replica exatamente a coluna
# composta já usada no check de qualidade "unique" (ver CHECKS acima).
CHAVE_NEGOCIO = {
    "uf": ["ano", "sigla_uf", "serie", "rede"],
    "municipio": ["ano", "id_municipio", "serie", "rede"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf", "rede"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "alunos": ["ano", "id_aluno"],  # CORRIGIDO: id_aluno sozinho não é único - se repete entre anos diferentes (descoberto rodando com dado real; ver diagnostico_fanout_alunos.sql)
    "pib_municipio": ["ano", "id_municipio"],
    "indicadores_educacionais_municipio": ["ano", "id_municipio"],
    "populacao_municipio": ["ano", "id_municipio"],
    "inse_escola": ["ano", "id_escola"],
}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# APLICA AS TRANSFORMAÇÕES COMUNS DA CAMADA SILVER
"""
    Args:
        df (pandas.DataFrame): DataFrame da Bronze.
        tabela (str): Nome da tabela.

    Nota sobre rastreabilidade:
        Os metadados técnicos da Bronze (_record_hash, _source_dataset,
        _source_table, _ingestion_timestamp, _ingestion_date) NÃO são
        removidos aqui. Eles atravessam a Silver propositalmente, para
        permitir auditoria de ponta a ponta (padrão Medalhão). A limpeza
        do schema analítico acontece na Gold, que já seleciona
        explicitamente suas colunas finais.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def construir_silver(df, tabela):

    log.info(f"Transformando tabela: {tabela}")

    df = df.copy()

    linhas_antes = len(df)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Remove espaços em branco
    # (metadados técnicos da Bronze são preservados - ver nota acima)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    colunas_texto = df.select_dtypes(include="object").columns

    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Conversão de tipo
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df = aplicar_conversao_tipo(df, tabela)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Normalização de chave (formato, não conteúdo de negócio)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df = normalizar_chaves_join(df, tabela)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Remove duplicidade por CHAVE DE NEGÓCIO COMPLETA (não pela linha
    # inteira, nem por um subconjunto de colunas). Como os metadados de
    # ingestão são preservados, duas cargas do mesmo registro de negócio
    # em datas diferentes teriam _ingestion_timestamp diferente e não
    # seriam pegas por um dedup de linha inteira. Por isso ordenamos por
    # _ingestion_timestamp e mantemos a versão mais recente por chave de
    # negócio.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    chave_negocio = CHAVE_NEGOCIO.get(tabela, [])
    chave_negocio = [c for c in chave_negocio if c in df.columns]

    if chave_negocio:

        if "_ingestion_timestamp" in df.columns:
            df = df.sort_values("_ingestion_timestamp")

        antes = len(df)
        df = df.drop_duplicates(subset=chave_negocio, keep="last")
        duplicados = antes - len(df)

        if duplicados > 0:
            log.info(
                f"[SILVER] {tabela}: {duplicados} registro(s) duplicado(s) "
                f"por chave de negócio {chave_negocio} removido(s) "
                f"(mantida a versão mais recente por _ingestion_timestamp)"
            )
    else:
        antes = len(df)
        df = df.drop_duplicates()
        duplicados = antes - len(df)
        if duplicados > 0:
            log.info(f"[SILVER] {tabela}: {duplicados} linha(s) duplicada(s) (linha inteira) removida(s)")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Trata nulos em colunas essenciais (chaves de negócio)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    colunas_essenciais = COLUNAS_ESSENCIAIS.get(tabela, [])
    colunas_essenciais = [c for c in colunas_essenciais if c in df.columns]

    if colunas_essenciais:

        nulos_antes = len(df)
        df = df.dropna(subset=colunas_essenciais)
        removidas = nulos_antes - len(df)

        if removidas > 0:
            log.warning(
                f"[SILVER] {tabela}: {removidas} linha(s) removida(s) por nulo "
                f"em coluna essencial {colunas_essenciais}"
            )

    log.info(
        f"[SILVER] {tabela}: {linhas_antes} linha(s) na entrada -> "
        f"{len(df)} linha(s) na saída"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Adiciona metadado da Silver
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df["_silver_processed_at"] = datetime.now()

    log.info(f"Tabela {tabela} transformada")

    return df

In [45]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# INTEGRAÇÃO ENTRE AS BASES: ALUNOS + MUNICÍPIO + UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Mapeamento oficial do IBGE: os 2 primeiros dígitos do código de 7 dígitos
# do município identificam a UF. Usado para derivar `sigla_uf` em `alunos`,
# já que a extração da Base dos Dados não trouxe essa coluna diretamente.
CODIGO_UF_PARA_SIGLA = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS",
    50: "MS", 51: "MT", 52: "GO", 53: "DF",
}


def derivar_sigla_uf(id_municipio):
    """
    Deriva a sigla da UF a partir do código IBGE do município.

    Args:
        id_municipio: Código IBGE do município (string ou int, 7 dígitos).

    Returns:
        str | None: Sigla da UF, ou None se o código for inválido/ausente.
    """
    try:
        codigo_uf = int(str(int(id_municipio))[:2])
        return CODIGO_UF_PARA_SIGLA.get(codigo_uf)
    except (ValueError, TypeError):
        return None


def construir_silver_alunos_integrado(df_alunos, df_municipio, df_uf):
    """
    Integra a tabela de alunos (granularidade individual) com indicadores
    agregados de município e de UF, adicionando contexto territorial a
    cada registro de aluno. Esta é a integração citada no README do
    projeto (join entre alunos, município e UF).

    Args:
        df_alunos (pandas.DataFrame): Tabela `alunos` já tratada na Silver.
        df_municipio (pandas.DataFrame): Tabela `municipio` já tratada na Silver.
        df_uf (pandas.DataFrame): Tabela `uf` já tratada na Silver.

    Returns:
        pandas.DataFrame: `alunos` enriquecido com contexto municipal e estadual.
    """

    log.info("Integrando alunos + município + UF")

    df = df_alunos.copy()

    # Deriva a UF do aluno a partir do código do município
    df["sigla_uf"] = df["id_municipio"].apply(derivar_sigla_uf)

    sem_uf = df["sigla_uf"].isnull().sum()
    if sem_uf > 0:
        log.warning(f"[SILVER] {sem_uf} aluno(s) sem sigla_uf derivada")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Contexto municipal (mesmo ano + município + rede)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    contexto_municipio = (
        df_municipio[["ano", "id_municipio", "rede", "taxa_alfabetizacao", "media_portugues"]]
        .rename(columns={
            "taxa_alfabetizacao": "taxa_alfabetizacao_municipio",
            "media_portugues": "media_portugues_municipio",
        })
    )

    df = df.merge(
        contexto_municipio,
        on=["ano", "id_municipio", "rede"],
        how="left",
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Contexto estadual (mesmo ano + UF + rede)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    contexto_uf = (
        df_uf[["ano", "sigla_uf", "sigla_uf_nome", "rede", "taxa_alfabetizacao", "media_portugues"]]
        .rename(columns={
            "taxa_alfabetizacao": "taxa_alfabetizacao_uf",
            "media_portugues": "media_portugues_uf",
        })
    )

    df = df.merge(
        contexto_uf,
        on=["ano", "sigla_uf", "rede"],
        how="left",
    )

    df["_silver_integrado_processed_at"] = datetime.now()

    log.info(
        f"Integração concluída: {len(df)} registro(s) de aluno, "
        f"{df['taxa_alfabetizacao_municipio'].isnull().sum()} sem contexto municipal, "
        f"{df['taxa_alfabetizacao_uf'].isnull().sum()} sem contexto estadual"
    )

    return df

In [46]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA SILVER EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
        tabela (str): Nome da tabela.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_silver(df, tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Tabela {tabela} da camada SILVER salva em {caminho}")

    return caminho

In [47]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUTA TODA A CAMADA SILVER PARA TODAS AS TABELAS DA BRONZE
"""
    Cada tabela é processada de forma isolada: se uma tabela falhar, um
    alerta é emitido e as demais continuam sendo processadas. A
    integração alunos+município+UF só é executada se as três tabelas
    de origem tiverem sido processadas com sucesso.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_silver():

    inicio_pipeline = time.perf_counter()

    tabelas_silver = {}
    tabelas_ok = 0
    tabelas_falha = 0

    for tabela in TABELAS:

        log.info("~" * 60)
        log.info(f"Iniciando camada SILVER: {tabela}")
        log.info("~" * 60)

        inicio_tabela = time.perf_counter()

        try:

            df = ler_bronze(tabela)

            df_silver = construir_silver(
                df,
                tabela
            )

            checks = CHECKS.get(tabela, [])

            if checks:
                checar_qualidade(df_silver, checks)

            salvar_silver(
                df_silver,
                tabela
            )

            tabelas_silver[tabela] = df_silver

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="silver",
                tabela=tabela,
                volume=len(df_silver),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            tabelas_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="silver",
                tabela=tabela,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha no processamento da tabela '{tabela}' na camada Silver",
                camada="silver",
                tabela=tabela,
                erro=str(e)
            )

            tabelas_falha += 1

            # isola a falha: segue para a próxima tabela
            continue

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Integração entre as bases (alunos + município + UF)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    tabelas_necessarias = {"alunos", "municipio", "uf"}

    if tabelas_necessarias.issubset(tabelas_silver.keys()):

        log.info("~" * 60)
        log.info("Iniciando integração: alunos + município + UF")
        log.info("~" * 60)

        inicio_integracao = time.perf_counter()

        try:

            df_alunos_integrado = construir_silver_alunos_integrado(
                tabelas_silver["alunos"],
                tabelas_silver["municipio"],
                tabelas_silver["uf"],
            )

            # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            # Validação entre tabelas: chaves de relacionamento e consistência
            # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            log.info("~" * 60)
            log.info("Validando chaves de relacionamento e consistência territorial")
            log.info("~" * 60)

            checar_integridade_referencial(
                df_alunos_integrado, "id_municipio",
                tabelas_silver["municipio"], "id_municipio",
                nome_relacao="municipio", critico=False,
            )

            checar_integridade_referencial(
                df_alunos_integrado, "sigla_uf",
                tabelas_silver["uf"], "sigla_uf",
                nome_relacao="uf", critico=False,
            )

            checar_consistencia_territorial(df_alunos_integrado, critico=False)

            salvar_silver(df_alunos_integrado, "alunos_integrado")

            latencia_segundos = round(time.perf_counter() - inicio_integracao, 2)

            log_metrica(
                "integracao_processada",
                camada="silver",
                tabela="alunos_integrado",
                volume=len(df_alunos_integrado),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_integracao, 2)

            log_metrica(
                "integracao_processada",
                camada="silver",
                tabela="alunos_integrado",
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                "Falha na integração alunos+município+UF na camada Silver",
                camada="silver",
                erro=str(e)
            )

            tabelas_falha += 1

    else:

        faltantes = tabelas_necessarias - tabelas_silver.keys()

        emitir_alerta(
            "Integração alunos+município+UF não executada: tabela(s) de origem ausente(s)",
            camada="silver",
            tabelas_faltantes=", ".join(sorted(faltantes))
        )

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="silver",
        tabelas_ok=tabelas_ok,
        tabelas_falha=tabelas_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if tabelas_falha > 0:
        emitir_alerta(
            f"Pipeline Silver concluído com {tabelas_falha} falha(s)",
            camada="silver",
            tabelas_falha=tabelas_falha,
            tabelas_ok=tabelas_ok
        )

    log.info("Camada SILVER concluída!")

In [48]:
executar_silver()

2026-08-25 07:39:59,633 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-25 07:39:59,634 | INFO     | Iniciando camada SILVER: uf
2026-08-25 07:39:59,635 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-25 07:39:59,636 | INFO     | Lendo a camada Bronze (partição mais recente): bronze\uf\ingestion_date=2026-08-25\uf.parquet
2026-08-25 07:39:59,769 | INFO     | Transformando tabela: uf
2026-08-25 07:39:59,781 | INFO     | [SILVER] uf: 145 linha(s) na entrada -> 145 linha(s) na saída
2026-08-25 07:39:59,782 | INFO     | Tabela uf transformada
2026-08-25 07:39:59,782 | INFO     | [DQ:SILVER] Iniciando verificações (5 regra(s))
2026-08-25 07:39:59,782 | INFO     | [DQ:SILVER] PASS | min_count | None | contagem=145 | mínimo=1
2026-08-25 07:39:59,783 | INFO     | [DQ:SILVER] PASS | not_null | ano | 0 nulos encontrados
2026-08-25 07:39:59,784 | INFO     | [DQ:SILVER] PASS | not_null | sigla_uf | 0 nulos encontrados
2026-08-25 0